# Проект: Анализ образовательных метрик

Продакт-менеджер Василий попросил вас проанализировать поведение студентов на курсах, предметах и экзаменах онлайн-школы и ответить на следующие вопросы:

 - 1. Сколько студентов успешно сдали только один курс? (Успешная сдача — это зачёт по курсу на экзамене)

 - 2. Найдите и отсортируйте id экзаменов в рамках курса по возрастанию уровня завершаемости*.

 - 3. Выявите самые популярные предметы (ТОП-3) по количеству регистраций на них.

 - 4. Выявите предметы с самым большим оттоком (ТОП-3).

 - 5. Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самой низкой завершаемостью курсов.

 - 6. Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самыми долгими средними сроками сдачи курсов.

 - 7. Часто для качественного анализа аудитории используют подходы, основанные на сегментации. Используя python, постройте адаптированные RFM-кластеры студентов, чтобы качественно оценить свою аудиторию. В адаптированной кластеризации можно выбрать следующие метрики:

R — среднее время сдачи одного экзамена,
F — завершаемость курсов,
M — среднее количество баллов, получаемое за экзамен.

Для каждого RFM-сегмента постройте границы метрик recency, frequency и monetary для интерпретации этих кластеров. Описание подхода можно найти тут.

- 7.1. Чему равна минимальная граница по recency?
- 7.2. Чему равна максимальная граница по recency?
- 7.3. Чему равна минимальная граница по monetary?
- 7.4. Чему равна максимальная граница по monetary?
- 7.5. Сколько клиентов попадут в кластер 232?

Используйте логику:
- a) по recency можно получить 2, если значение recency меньше либо равно,  медиане по recency. В остальных случаях — 1.
- b) по frequency можно получить 1, если значение frequency меньше 50. Можно получить 2, если значение по frequency меньше 100. Можно получить 3 в остальных случаях.
- c) по monetary можно получить 1, если значение monetary меньше 40. Можно получить 2, если значение по monetary меньше либо равно 80. Можно получить 3 в остальных случаях.

Для решения задачи проведите предварительное исследование данных и сформулируйте, что должно считаться курсом. Обосновать свой выбор вы можете с помощью фактов сдачи экзаменов, распределения студентов и уникального идентификатора курса.

*завершаемость = кол-во успешных экзаменов/кол-во всех попыток сдать экзамен * 100.

In [3]:
# импорт бибилиотек
import pandas as pd

In [4]:
# файл содержит информацию об оценках в тесте. Обычно каждый предмет в семестре включает ряд тестов с оценками,
# за которыми следует заключительный экзаменационный тест (экзамен).

assessments = pd.read_csv('assessments.csv')

assessments.head(10)

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0
5,AAA,2013J,1757,Exam,NaN,100.0
6,AAA,2014J,1758,TMA,19.0,10.0
7,AAA,2014J,1759,TMA,54.0,20.0
8,AAA,2014J,1760,TMA,117.0,20.0
9,AAA,2014J,1761,TMA,166.0,20.0


In [5]:
assessments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    object 
 1   code_presentation  206 non-null    object 
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    object 
 4   date               195 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 9.8+ KB


In [6]:
#  файл содержит список предметов по семестрам.

courses = pd.read_csv('courses.csv')

courses.head()

,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


In [7]:
courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   code_module                 22 non-null     object
 1   code_presentation           22 non-null     object
 2   module_presentation_length  22 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 656.0+ bytes


In [8]:
# файл содержит результаты тестов студентов.
# Если учащийся не отправляет работу на оценку, результат не записывается в таблицу.

student_assessment = pd.read_csv('studentAssessment.csv')

student_assessment.head()

,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [9]:
student_assessment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id_assessment   173912 non-null  int64  
 1   id_student      173912 non-null  int64  
 2   date_submitted  173912 non-null  int64  
 3   is_banked       173912 non-null  int64  
 4   score           173739 non-null  float64
dtypes: float64(1), int64(4)
memory usage: 6.6 MB


In [10]:
# этот файл содержит информацию о времени,
# когда студент зарегистрировался для прохождения курса в семестре.

student_registration = pd.read_csv('studentRegistration.csv')

student_registration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [11]:
student_registration.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   code_module          32593 non-null  object 
 1   code_presentation    32593 non-null  object 
 2   id_student           32593 non-null  int64  
 3   date_registration    32548 non-null  float64
 4   date_unregistration  10072 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.2+ MB


In [12]:
# Для решения задачи проведите предварительное исследование данных и сформулируйте,
# что должно считаться курсом. Обосновать свой выбор вы можете с помощью фактов сдачи экзаменов,
# распределения студентов и уникального идентификатора курса.

In [13]:
assessments_exam = assessments[assessments['assessment_type'] =='Exam']
assessments_exam

,code_module,code_presentation,id_assessment,assessment_type,date,weight
5,AAA,2013J,1757,Exam,NaN,100.0
11,AAA,2014J,1763,Exam,NaN,100.0
23,BBB,2013B,14990,Exam,NaN,100.0
35,BBB,2013J,15002,Exam,NaN,100.0
47,BBB,2014B,15014,Exam,NaN,100.0
53,BBB,2014J,15025,Exam,NaN,100.0
62,CCC,2014B,24290,Exam,NaN,100.0
63,CCC,2014B,40087,Exam,NaN,100.0
72,CCC,2014J,24299,Exam,NaN,100.0
73,CCC,2014J,40088,Exam,NaN,100.0


In [14]:
assessments_exam.groupby(['code_module', 'code_presentation'], as_index=False).agg({'id_assessment':'nunique'})

,code_module,code_presentation,id_assessment
0,AAA,2013J,1
1,AAA,2014J,1
2,BBB,2013B,1
3,BBB,2013J,1
4,BBB,2014B,1
5,BBB,2014J,1
6,CCC,2014B,2
7,CCC,2014J,2
8,DDD,2013B,1
9,DDD,2013J,1


**Кейс 1**

Сколько студентов успешно сдали только один курс?
(для подсчёта числа успешно сданных экзаменов нужно использовать id_assessment)

In [15]:
student_assessment_merge = assessments.merge(student_assessment, on='id_assessment')
student_assessment_merge.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
0,AAA,2013J,1752,TMA,19.0,10.0,11391,18,0,78.0
1,AAA,2013J,1752,TMA,19.0,10.0,28400,22,0,70.0
2,AAA,2013J,1752,TMA,19.0,10.0,31604,17,0,72.0
3,AAA,2013J,1752,TMA,19.0,10.0,32885,26,0,69.0
4,AAA,2013J,1752,TMA,19.0,10.0,38053,19,0,79.0


In [16]:
student_exam_done = student_assessment_merge.query("assessment_type == 'Exam' and score >= 40.0")
student_exam_done.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
52924,CCC,2014B,24290,Exam,NaN,100.0,559706,234,0,78.0
52925,CCC,2014B,24290,Exam,NaN,100.0,559770,230,0,54.0
52926,CCC,2014B,24290,Exam,NaN,100.0,560114,230,0,64.0
52927,CCC,2014B,24290,Exam,NaN,100.0,560311,234,0,100.0
52928,CCC,2014B,24290,Exam,NaN,100.0,560494,230,0,92.0


In [17]:
student_exam = student_exam_done.groupby('id_student', as_index=False).agg({'id_assessment':'nunique'})
student_exam

,id_student,id_assessment
0,23698,1
1,24213,1
2,27116,1
3,28046,1
4,28787,1
...,...,...
4092,2694886,1
4093,2694933,1
4094,2695608,1
4095,2697181,1


In [18]:
student_exam['id_assessment'].value_counts()

1    3802
2     295
Name: id_assessment, dtype: int64

In [19]:
student_exam[student_exam['id_assessment'] == 1].id_student.nunique()

3802

**Кейс 2**

Найдите и отсортируйте id экзаменов в рамках курса по возрастанию уровня завершаемости (1 - самый маленький, 6 - самый большой уровень завершаемости):

Давайте вспомним определение завершаемости из описания кейса: завершаемость= кол-во успешных экзаменов /кол-во всех попыток сдать экзамен ×100

При вашем подходе вы упускаете знаменатель этой формулы — общее количество попыток сдать каждый конкретный экзамен. Если на один экзамен пришло 100 человек и сдали 50, а на другой пришло 10 человек и сдали все 10, то по числу сдавших выиграет первый (50 против 10), а по уровню завершаемости — второй (100% против 50%).

Чтобы решить эту задачу верно, попробуйте сделать следующее:

Выделите только экзамены (в таблице с оценками их тип обычно обозначается как Exam).
Для каждого id_assessment (экзамена) найдите общее количество попыток сдачи (это просто общее число строк/записей для данного экзамена).
Найдите количество успешных попыток (где оценка score больше или равна 40 баллов).
Поделите количество успешных попыток на общее количество попыток и умножьте на 100, чтобы получить процент завершаемости.
И уже полученную итоговую метрику завершаемости отсортируйте с помощью sort_values().

In [20]:
student_exam_alltry = student_assessment_merge.query("assessment_type == 'Exam'") \
                        .groupby('id_assessment', as_index=False).agg({'score':'count'}) \
                        .rename(columns = {'score':'alltry'})
student_exam_alltry.head()

,id_assessment,alltry
0,24290,747
1,24299,1168
2,25340,602
3,25354,968
4,25361,524


In [21]:
student_exam_donetry = student_assessment_merge.query("assessment_type == 'Exam' and score >= 40.0") \
                        .groupby('id_assessment', as_index=False).agg({'score':'count'}) \
                        .rename(columns = {'score':'donetry'})
student_exam_donetry.head()

,id_assessment,donetry
0,24290,664
1,24299,1019
2,25340,504
3,25354,878
4,25361,485


In [22]:
student_exam_try = student_exam_alltry.merge(student_exam_donetry, on='id_assessment')
student_exam_try['done_rate'] = student_exam_try['donetry']/student_exam_try['alltry']*100
student_exam_try.sort_values(by='done_rate')

,id_assessment,alltry,donetry,done_rate
2,25340,602,504,83.720930
1,24299,1168,1019,87.243151
5,25368,950,842,88.631579
0,24290,747,664,88.888889
3,25354,968,878,90.702479
4,25361,524,485,92.557252


**Кейс 3**

Выявите самые популярные предметы (ТОП-3) по количеству регистраций на них:

In [23]:
student_registration.head()

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


In [24]:
student_registration.groupby('code_module', as_index=False).agg({'id_student':'nunique'})\
                    .sort_values(by='id_student', ascending=False).head(3)

,code_module,id_student
1,BBB,7692
5,FFF,7397
3,DDD,5848


**Кейс 4**

Выявите предметы с самым большим оттоком* (ТОП-3):
*Отток в данном случае — это отмена регистрации на предмет.
Считайте его в абсолютных значениях (т.е. количество людей, отменивших регистрацию).

In [25]:
student_registration.isnull().sum()

code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

In [26]:
# Удаление строк, где отсутствует 'date_unregistration'
student_registration_churn = student_registration.dropna(subset=['date_unregistration'], how='all')
student_registration_churn.isnull().sum()

code_module             0
code_presentation       0
id_student              0
date_registration      39
date_unregistration     0
dtype: int64

In [27]:
student_registration_churn.groupby('code_module', as_index=False).agg({'id_student':'nunique'}) \
                            .sort_values(by='id_student', ascending=False).head(3)

,code_module,id_student
1,BBB,2314
5,FFF,2249
3,DDD,2065


**Кейс 5**

Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самой низкой завершаемостью курсов:

In [28]:
student_assessment_merge.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
0,AAA,2013J,1752,TMA,19.0,10.0,11391,18,0,78.0
1,AAA,2013J,1752,TMA,19.0,10.0,28400,22,0,70.0
2,AAA,2013J,1752,TMA,19.0,10.0,31604,17,0,72.0
3,AAA,2013J,1752,TMA,19.0,10.0,32885,26,0,69.0
4,AAA,2013J,1752,TMA,19.0,10.0,38053,19,0,79.0


In [29]:
student_assessment_merge['code_presentation'].value_counts()

2014J    52174
2013J    49018
2014B    38896
2013B    33824
Name: code_presentation, dtype: int64

In [30]:
student_sem_alltry = student_assessment_merge.query("assessment_type == 'Exam'") \
                        .groupby('code_presentation', as_index=False).agg({'score':'count'}) \
                        .rename(columns = {'score':'alltry'})
student_sem_alltry

,code_presentation,alltry
0,2013B,602
1,2013J,968
2,2014B,1271
3,2014J,2118


In [31]:
student_sem_donetry = student_assessment_merge.query("assessment_type == 'Exam' and score >= 40.0") \
                        .groupby('code_presentation', as_index=False).agg({'score':'count'}) \
                        .rename(columns = {'score':'donetry'})
student_sem_donetry.head()

,code_presentation,donetry
0,2013B,504
1,2013J,878
2,2014B,1149
3,2014J,1861


In [32]:
student_sem_try = student_sem_alltry.merge(student_sem_donetry, on='code_presentation')
student_sem_try['done_rate'] = student_sem_try['donetry']/student_sem_try['alltry']*100
student_sem_try.sort_values(by='done_rate')

,code_presentation,alltry,donetry,done_rate
0,2013B,602,504,83.720930
3,2014J,2118,1861,87.865911
2,2014B,1271,1149,90.401259
1,2013J,968,878,90.702479


**Кейс 6**

Используя pandas, в период с начала 2013 по конец 2014 выявите семестр с самыми долгими средними сроками сдачи курсов:

In [33]:
student_assessment_merge.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
0,AAA,2013J,1752,TMA,19.0,10.0,11391,18,0,78.0
1,AAA,2013J,1752,TMA,19.0,10.0,28400,22,0,70.0
2,AAA,2013J,1752,TMA,19.0,10.0,31604,17,0,72.0
3,AAA,2013J,1752,TMA,19.0,10.0,32885,26,0,69.0
4,AAA,2013J,1752,TMA,19.0,10.0,38053,19,0,79.0


In [34]:
student_assessment_merge.query("assessment_type == 'Exam'") \
                        .groupby('code_presentation', as_index=False).agg({'date_submitted':'mean'}) \
                        .rename(columns = {'date_submitted':'date_submitted_mean'})\
                        .sort_values(by='date_submitted_mean', ascending = False)

,code_presentation,date_submitted_mean
3,2014J,243.674221
1,2013J,239.505165
2,2014B,232.988985
0,2013B,230.179402


In [35]:
student_assessment_merge.query("assessment_type == 'Exam' and score >= 40.0") \
                        .groupby('code_presentation', as_index=False).agg({'date_submitted':'mean'}) \
                        .rename(columns = {'date_submitted':'date_submitted_mean'})\
                        .sort_values(by='date_submitted_mean', ascending = False)

,code_presentation,date_submitted_mean
3,2014J,243.680279
1,2013J,239.509112
2,2014B,232.997389
0,2013B,230.164683


**Кейс 7-10**

Часто для качественного анализа аудитории используют подходы, основанные на сегментации.
Используя python, постройте адаптированные RFM-кластеры студентов, чтобы качественно оценить свою аудиторию.

В адаптированной кластеризации можете выбрать следующие метрики:
- R — среднее время сдачи одного экзамена,
- F — завершаемость курсов,
- M — среднее количество баллов, получаемое за экзамен.

Для каждого RFM-сегмента постройте границы метрик recency, frequency и monetary для интерпретации этих кластеров.

Чему равна минимальная граница по recency?

In [36]:
recency_df = student_assessment_merge.query("assessment_type == 'Exam'") \
                        .groupby('id_student').agg({'date_submitted':'mean'}) \
                        .rename(columns = {'date_submitted':'recency'})
recency_df.head()

,recency
id_student,
23698,243.0
24213,236.0
27116,243.0
28046,237.0
28787,243.0


In [37]:
recency_df.describe()

,recency
count,4633.000000
mean,238.462227
std,5.653378
min,229.000000
25%,234.000000
50%,241.000000
75%,243.000000
max,285.000000


In [38]:
monetary_df = student_assessment_merge.query("assessment_type == 'Exam'") \
                        .groupby('id_student').agg({'score':'mean'}) \
                        .rename(columns = {'score':'monetary'})
monetary_df.head()

,monetary
id_student,
23698,80.0
24213,58.0
27116,96.0
28046,40.0
28787,44.0


In [39]:
monetary_df.describe()

,monetary
count,4633.000000
mean,65.117958
std,20.470561
min,0.000000
25%,50.000000
50%,66.000000
75%,82.000000
max,100.000000


In [40]:
student_assessment_merge.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight,id_student,date_submitted,is_banked,score
0,AAA,2013J,1752,TMA,19.0,10.0,11391,18,0,78.0
1,AAA,2013J,1752,TMA,19.0,10.0,28400,22,0,70.0
2,AAA,2013J,1752,TMA,19.0,10.0,31604,17,0,72.0
3,AAA,2013J,1752,TMA,19.0,10.0,32885,26,0,69.0
4,AAA,2013J,1752,TMA,19.0,10.0,38053,19,0,79.0


In [41]:
frequency_alltry = student_assessment_merge.query("assessment_type == 'Exam'") \
                        .groupby('id_student').agg({'score':'count'}) \
                        .rename(columns = {'score':'alltry'})
frequency_alltry.head()

,alltry
id_student,
23698,1
24213,1
27116,1
28046,1
28787,1


In [42]:
frequency_donetry = student_assessment_merge.query("assessment_type == 'Exam' and score >= 40.0") \
                        .groupby('id_student').agg({'score':'count'}) \
                        .rename(columns = {'score':'donetry'})
frequency_donetry.head()

,donetry
id_student,
23698,1
24213,1
27116,1
28046,1
28787,1


In [43]:
frequency_df = frequency_alltry.merge(frequency_donetry, on='id_student')
frequency_df['frequency'] = frequency_df['donetry']/frequency_df['alltry']*100
frequency_df.describe()

,alltry,donetry,frequency
count,4097.000000,4097.000000,4097.000000
mean,1.078838,1.072004,99.658287
std,0.269519,0.258526,4.119836
min,1.000000,1.000000,50.000000
25%,1.000000,1.000000,100.000000
50%,1.000000,1.000000,100.000000
75%,1.000000,1.000000,100.000000
max,2.000000,2.000000,100.000000


In [44]:
freq_df = frequency_df[['frequency']]
freq_df.head()

,frequency
id_student,
23698,100.0
24213,100.0
27116,100.0
28046,100.0
28787,100.0


In [45]:
rfm = pd.concat([recency_df, freq_df, monetary_df], axis=1).reset_index()
rfm.head()

,id_student,recency,frequency,monetary
0,23698,243.0,100.0,80.0
1,24213,236.0,100.0,58.0
2,27116,243.0,100.0,96.0
3,28046,237.0,100.0,40.0
4,28787,243.0,100.0,44.0


**Кейс 11**

Используя логику:

- 1) по recency можно получить 2, если значение recency меньше либо равно медиане по recency. В остальных случаях — 1.

- 2) по frequency можно получить 1, если значение frequency меньше 50.
Можно получить 2, если значение по frequency меньше 100.
Можно получить 3 в остальных случаях.

- 3) по monetary можно получить 1, если значение monetary меньше 40.
Можно получить 2, если значение по monetary меньше либо равно 80.
Можно получить 3 в остальных случаях.

Сколько клиентов попадут в кластер 232?

In [46]:
# Чем меньше среднее время сдачи экзамена (recency), тем выше балл
rfm['R_Score'] = pd.qcut(rfm['recency'], 2, labels=[2, 1])  

In [47]:
# Чем выше завершаемость курсов (frequency), тем выше балл
rfm['F_Score'] = pd.cut(rfm['frequency'], bins=[-float('inf'), 49, 99, float('inf')], labels=[1, 2, 3]) 

In [48]:
 # Чем выше средний балл за экзамен (monetary), тем выше балл
rfm['M_Score'] = pd.cut(rfm['monetary'], bins=[-float('inf'), 39, 80, float('inf')], labels=[1, 2, 3])  

In [49]:
# Объединение RFM баллов в один общий балл
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm.head()

,id_student,recency,frequency,monetary,R_Score,F_Score,M_Score,RFM_Score
0,23698,243.0,100.0,80.0,1,3,2,132
1,24213,236.0,100.0,58.0,2,3,2,232
2,27116,243.0,100.0,96.0,1,3,3,133
3,28046,237.0,100.0,40.0,2,3,2,232
4,28787,243.0,100.0,44.0,1,3,2,132


In [50]:
# Сколько клиентов попадут в кластер 232?

rfm[rfm['RFM_Score'] == '232'].id_student.nunique()

1555